In [1]:
from openai import OpenAI
import json
from tqdm import tqdm, trange

In [ ]:
key=''
client = OpenAI(api_key=key)
models = client.models.list()

In [3]:
# sample_data = []
with open('/kaggle/input/medqa-us-data/final_augment_test_questions_v2.json', 'r') as f:
    dataset = json.loads(f.read())
    f.close()

In [4]:
system_prompt = """You are a medical expert assistant.
You are given multiple-choice medical questions.
Your task is to choose the single best answer.

OUTPUT CONSTRAINTS (STRICT):
- Do NOT provide any explanation, reasoning, or commentary.
- Do NOT include any words, symbols, or punctuation.
- Do NOT include quotes, prefixes, or suffixes.
- Do NOT include extra whitespace or newlines.
- The response must consist of exactly one character.

PENALTIES FOR VIOLATION:
- Any output longer than one character will be considered a critical error.
- Any output not exactly one of (A, B, C, D, or E) will receive a zero score.

FINAL FORMAT (MANDATORY):
X
where X is one of (A, B, C, D, or E)."""


user_prompt = """Question:
{question}

Options:
{options}

Select the single best answer.
Output exactly one character and nothing else:
A, B, C, D, or E."""

In [5]:
def form_message(question_text, options_dict):
  chat = {'conversations': []}
  chat['conversations'].append({
        "role": "system",
        "content": system_prompt
  })


  options_str = ''
  for k, v in options_dict.items():
      options_str += f'{k}. {v}\n'

  chat['conversations'].append({
        "role": "user",
        "content":user_prompt.format(question=question_text,
                               options=options_str)
    })
    
  return chat

In [6]:
all_chats = []
for d in dataset:
    chats = {}
    chats['original'] = form_message(d['original_question'], d['options'])['conversations']
    chats['neutral'] = form_message(d['neutral_question'], d['options'])['conversations']
    for t in ['t1', 't2', 't3']:
        chat_obj = {}
        for e in ['i', 'c', 'ic']:
            chat_obj[e] = form_message(d[t][e], d['options'])['conversations']
        chats[t] = chat_obj.copy()
            
    all_chats.append(chats)

In [7]:
def get_response(client, messages):
    completion = client.chat.completions.create(
        model="gpt-5.2",
        messages=messages,
        temperature = 0.0
    )
    return completion.choices[0].message.content

In [8]:
# get_response(client, all_chats[0]['neutral'])

In [9]:
from tqdm import trange, tqdm
import json

all_answers = []
for chats in tqdm(all_chats):
    ans_obj = {}
    ans_obj['original'] = get_response(client, chats['original'])
    ans_obj['neutral'] = get_response(client, chats['neutral'])
    for t in ['t1', 't2', 't3']:
        ans_obj[t] = {}
        for e in ['i', 'c', 'ic']:
            ans_obj[t][e] = get_response(client, chats[t][e])
    all_answers.append(ans_obj)
    
    if len(all_answers) % 10 == 0:
        with open('GPT5.2_option_augment_answers.json', 'w') as f:
            f.write(json.dumps(all_answers))
            f.close()

100%|██████████| 150/150 [17:49<00:00,  7.13s/it]


In [10]:
with open('GPT5.2_option_final_augment_answers.json', 'w') as f:
            f.write(json.dumps(all_answers))
            f.close()